In [ ]:
import os
import pandas as pd

folder = r"C:\Users\DELL\Documents\project_data\data"
file_path = os.path.join(folder, "dirty_cafe_sales.csv")

df = pd.read_csv(file_path)
# -----------------------------
# 1. Fix column names
# -----------------------------
df.columns = df.columns.str.strip()
print(df.head())
# -----------------------------
# 2. Replace bad values
# -----------------------------
df.replace("ERROR", pd.NA, inplace=True)
df.replace("UNKNOWN", pd.NA, inplace=True)

# -----------------------------
# 3. Convert numeric columns
# -----------------------------
num_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 4. Fix Total Spent if missing
# -----------------------------
df["Total Spent"] = df["Total Spent"].fillna(
    df["Quantity"] * df["Price Per Unit"]
)

# -----------------------------
# 5. Fill numeric nulls with median
# -----------------------------
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# -----------------------------
# 6. Fix categorical columns
# -----------------------------
cat_cols = ["Item", "Payment Method", "Location"]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# -----------------------------
# 7. Convert date column
# -----------------------------
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

# -----------------------------
# 8. Create extra BI columns
# -----------------------------
df["Year"] = df["Transaction Date"].dt.year
df["Month"] = df["Transaction Date"].dt.month
df["Day"] = df["Transaction Date"].dt.day
df["Weekday"] = df["Transaction Date"].dt.day_name()

# Fill missing dates with the most frequent date (mode)
mode_date = df["Transaction Date"].mode()[0]
df["Transaction Date"] = df["Transaction Date"].fillna(mode_date)

# Recreate time columns correctly as integers
df["Year"] = df["Transaction Date"].dt.year.astype(int)
df["Month"] = df["Transaction Date"].dt.month.astype(int)
df["Day"] = df["Transaction Date"].dt.day.astype(int)
df["Weekday"] = df["Transaction Date"].dt.day_name()
# -----------------------------
# 9. Save clean file
# -----------------------------
#df.to_csv("transactions_clean.csv", index=False)

print("Clean dataset ready for Power BI ✅")

  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  
